## 特征计算

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from PIL import Image

# 1) 读取标签表，并提取 image_id 开头的数字
df_label = pd.read_csv('label.csv')  # 列: image_id, dx

def extract_id(s):
    m = re.match(r'^(\d+)', str(s))
    return int(m.group(1)) if m else None

df_label['clean_id'] = df_label['image_id'].apply(extract_id)
# 丢弃无法提取数字的行（避免后面出错）
df_label = df_label.dropna(subset=['clean_id']).copy()

# 使用清洗后的 clean_id 构造字典
id_to_label = dict(zip(df_label['clean_id'].astype(int), df_label['dx']))

# 2) 遍历 image 目录并建立 (路径, 标签) 对
img_dir = Path('image')
img_paths = sorted([p for p in img_dir.glob('*.jpg')])

samples = []
for p in img_paths:
    m = re.match(r'^(\d+)', p.stem)
    if m is None:
        continue
    image_id = int(m.group(1))
    if image_id in id_to_label:
        samples.append((p, id_to_label[image_id]))

print(f'匹配到 {len(samples)} 张带标签图片')

# 3) 把字符串标签编码为整数标签
classes = sorted(df_label['dx'].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}

# 4) 实际读取图片，构建训练集
X_train, y_train = [], []
for img_path, label_str in samples:
    img = Image.open(img_path).convert('RGB')
    X_train.append(np.array(img))
    y_train.append(class_to_idx[label_str])

print(f'类别映射: {class_to_idx}')
print(f'训练样本数: {len(X_train)}')
if X_train:
    print(f'第1张图尺寸: {X_train[0].shape}')
else:
    print('没有加载到任何图片，请检查目录和标签匹配情况')

匹配到 600 张带标签图片
类别映射: {'mel': 0, 'nv': 1, 'vasc': 2}
训练样本数: 600
第1张图尺寸: (224, 224, 3)


In [8]:
import os
import cv2
import numpy as np
import pandas as pd
from skimage.feature import graycomatrix, graycoprops

def compute_glcm_features(image_path, mask_path, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4], levels=64):
    """
    计算单张图像的 GLCM 特征（基础8个 + 扩展6个），均取四个方向平均
    返回字典，包含：
        ASM, Contrast, Correlation, Variance, Homogeneity, Entropy, MaxProbability, Dissimilarity,
        SumVariance, SumEntropy, DifferenceVariance, DifferenceEntropy, IMCorr1, IMCorr2
    """
    # 读取图像并转为灰度
    img = cv2.imread(image_path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"无法读取图像: {image_path}")
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # 读取 mask
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise FileNotFoundError(f"无法读取mask: {mask_path}")
    _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    
    # 找到 mask 包围盒，裁剪 ROI
    coords = cv2.findNonZero(mask)
    if coords is None:
        # 空 mask，返回全 NaN
        nan_dict = {k: np.nan for k in ['ASM', 'Contrast', 'Correlation', 'Variance', 'Homogeneity', 
                                        'Entropy', 'MaxProbability', 'Dissimilarity',
                                        'SumVariance', 'SumEntropy', 'DifferenceVariance', 
                                        'DifferenceEntropy', 'IMCorr1', 'IMCorr2']}
        return nan_dict
    
    x, y, w, h = cv2.boundingRect(coords)
    roi_gray = gray[y:y+h, x:x+w]
    # 灰度量化
    roi_gray = (roi_gray / 256 * levels).astype(np.uint8)
    roi_gray[roi_gray >= levels] = levels - 1
    
    # 计算 GLCM（四个方向，距离=1）
    glcm = graycomatrix(roi_gray, distances=distances, angles=angles, levels=levels,
                        symmetric=True, normed=True)
    # glcm 形状: (levels, levels, len(distances), len(angles))
    n_angles = len(angles)
    
    # 初始化各特征累积列表
    asm_list = []
    contrast_list = []
    correlation_list = []
    homogeneity_list = []
    dissimilarity_list = []
    entropy_list = []
    maxprob_list = []
    variance_list = []
    
    sum_var_list = []
    sum_ent_list = []
    diff_var_list = []
    diff_ent_list = []
    imcorr1_list = []
    imcorr2_list = []
    
    for i in range(n_angles):
        glcm_angle = glcm[:, :, 0, i]  # (levels, levels)
        # ----- 基础特征（使用 skimage 提供的）-----
        # 注意：graycoprops 需要传入 4D 数组，这里为单个方向单独构造
        glcm_4d = glcm_angle.reshape(levels, levels, 1, 1)
        asm_list.append(graycoprops(glcm_4d, 'ASM')[0,0])
        contrast_list.append(graycoprops(glcm_4d, 'contrast')[0,0])
        correlation_list.append(graycoprops(glcm_4d, 'correlation')[0,0])
        homogeneity_list.append(graycoprops(glcm_4d, 'homogeneity')[0,0])
        dissimilarity_list.append(graycoprops(glcm_4d, 'dissimilarity')[0,0])
        
        # ----- 手动计算熵、最大概率、方差 -----
        p = glcm_angle + 1e-12
        entropy = -np.sum(p * np.log(p))
        entropy_list.append(entropy)
        maxprob_list.append(np.max(glcm_angle))
        # 方差：Var = sum (i - mu)^2 * P(i,j)
        px = np.sum(glcm_angle, axis=1)  # 边缘概率 P_x(i)
        mu = np.sum(np.arange(levels) * px)
        var = np.sum((np.arange(levels)[:, None] - mu)**2 * glcm_angle)
        variance_list.append(var)
        
        # ----- 扩展特征：需要和概率、差概率 -----
        # 和概率 P_{x+y}(k), k = 2,3,...,2*levels
        sum_prob = np.zeros(2*levels + 1)
        for i_idx in range(levels):
            for j_idx in range(levels):
                k = i_idx + j_idx
                sum_prob[k] += glcm_angle[i_idx, j_idx]
        # 有效 k 范围 2 到 2*levels
        k_vals = np.arange(2, 2*levels+1)
        sum_prob_valid = sum_prob[2:2*levels+1]  # 长度 2*levels-1
        # 和均值
        mu_sum = np.sum(k_vals * sum_prob_valid)
        # 和方差
        sum_var = np.sum((k_vals - mu_sum)**2 * sum_prob_valid)
        sum_var_list.append(sum_var)
        # 和熵
        sum_ent = -np.sum(sum_prob_valid * np.log(sum_prob_valid + 1e-12))
        sum_ent_list.append(sum_ent)
        
        # 差概率 P_{x-y}(k), k = 0,1,...,levels-1
        diff_prob = np.zeros(levels)
        for i_idx in range(levels):
            for j_idx in range(levels):
                k = abs(i_idx - j_idx)
                diff_prob[k] += glcm_angle[i_idx, j_idx]
        # 差均值
        mu_diff = np.sum(np.arange(levels) * diff_prob)
        # 差方差
        diff_var = np.sum((np.arange(levels) - mu_diff)**2 * diff_prob)
        diff_var_list.append(diff_var)
        # 差熵
        diff_ent = -np.sum(diff_prob * np.log(diff_prob + 1e-12))
        diff_ent_list.append(diff_ent)
        
        # IMCorr1, IMCorr2 需要更多熵量
        # 边缘概率
        px = np.sum(glcm_angle, axis=1)
        py = np.sum(glcm_angle, axis=0)
        # H(X), H(Y)
        HX = -np.sum(px * np.log(px + 1e-12))
        HY = -np.sum(py * np.log(py + 1e-12))
        # H(XY)
        HXY = -np.sum(glcm_angle * np.log(glcm_angle + 1e-12))
        # HXY1 = -sum P(i,j) log[Px(i) Py(j)]
        HXY1 = -np.sum(glcm_angle * np.log(px[:, None] * py[None, :] + 1e-12))
        # HXY2 = -sum Px(i) Py(j) log[Px(i) Py(j)]
        HXY2 = -np.sum(px[:, None] * py[None, :] * np.log(px[:, None] * py[None, :] + 1e-12))
        # IMCorr1
        imcorr1 = (HXY - HXY1) / max(HX, HY)
        imcorr1_list.append(imcorr1)
        # IMCorr2
        imcorr2 = np.sqrt(1 - np.exp(-2 * (HXY2 - HXY)))
        imcorr2_list.append(imcorr2)
    
    # 对每个特征取四个方向平均
    features = {
        'ASM': np.mean(asm_list),
        'Contrast': np.mean(contrast_list),
        'Correlation': np.mean(correlation_list),
        'Homogeneity': np.mean(homogeneity_list),
        'Dissimilarity': np.mean(dissimilarity_list),
        'Entropy': np.mean(entropy_list),
        'MaxProbability': np.mean(maxprob_list),
        'Variance': np.mean(variance_list),
        'SumVariance': np.mean(sum_var_list),
        'SumEntropy': np.mean(sum_ent_list),
        'DifferenceVariance': np.mean(diff_var_list),
        'DifferenceEntropy': np.mean(diff_ent_list),
        'IMCorr1': np.mean(imcorr1_list),
        'IMCorr2': np.mean(imcorr2_list)
    }
    return features

def process_all_pairs(image_dir, mask_dir, output_csv='glcm_features_full.csv'):
    """
    处理所有配对图像，输出 CSV 文件，包含文件名和各 GLCM 特征（基础+扩展）
    """
    all_images = [f for f in os.listdir(image_dir) if f.endswith('.jpg') and not f.startswith('mask_')]
    # 按数字排序
    import re
    def extract_number(name):
        match = re.search(r'(\d+)', name)
        return int(match.group(1)) if match else 0
    all_images.sort(key=extract_number)
    
    rows = []
    for img_file in all_images:
        img_path = os.path.join(image_dir, img_file)
        mask_file = 'mask_' + img_file
        mask_path = os.path.join(mask_dir, mask_file)
        if not os.path.exists(mask_path):
            print(f"警告: 找不到对应mask {mask_path}，跳过图片 {img_file}")
            row = {'filename': img_file, 'ASM': np.nan, 'Contrast': np.nan, 'Correlation': np.nan,
                   'Variance': np.nan, 'Homogeneity': np.nan, 'Entropy': np.nan,
                   'MaxProbability': np.nan, 'Dissimilarity': np.nan,
                   'SumVariance': np.nan, 'SumEntropy': np.nan, 'DifferenceVariance': np.nan,
                   'DifferenceEntropy': np.nan, 'IMCorr1': np.nan, 'IMCorr2': np.nan}
        else:
            try:
                feats = compute_glcm_features(img_path, mask_path)
                row = {'filename': img_file, **feats}
                print(f"处理完成: {img_file} -> ASM={feats['ASM']:.6f}, Contrast={feats['Contrast']:.6f}")
            except Exception as e:
                print(f"处理 {img_file} 时出错: {e}")
                row = {'filename': img_file, 'ASM': np.nan, 'Contrast': np.nan, 'Correlation': np.nan,
                       'Variance': np.nan, 'Homogeneity': np.nan, 'Entropy': np.nan,
                       'MaxProbability': np.nan, 'Dissimilarity': np.nan,
                       'SumVariance': np.nan, 'SumEntropy': np.nan, 'DifferenceVariance': np.nan,
                       'DifferenceEntropy': np.nan, 'IMCorr1': np.nan, 'IMCorr2': np.nan}
        rows.append(row)
    
    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    print(f"特征已保存至 {output_csv}，共 {len(df)} 条记录，列数 {df.shape[1]}")
    return df

if __name__ == "__main__":
    # 请修改为实际路径
    image_directory = "image"   # 存放所有 jpg 图片的文件夹
    mask_directory = "mask"     # 存放对应 mask 的文件夹
    output_file = "glcm_features_full_processed.csv"
    df_results = process_all_pairs(image_directory, mask_directory, output_file)
    print("完成。")

处理完成: 1.jpg -> ASM=0.013980, Contrast=2.905996
处理完成: 1_aug1.jpg -> ASM=0.009896, Contrast=5.776674
处理完成: 1_aug2.jpg -> ASM=0.016810, Contrast=1.876917
处理完成: 2.jpg -> ASM=0.039199, Contrast=1.047067
处理完成: 2_aug1.jpg -> ASM=0.046070, Contrast=2.035261
处理完成: 2_aug2.jpg -> ASM=0.044356, Contrast=5.128638
处理完成: 3.jpg -> ASM=0.007032, Contrast=4.487863
处理完成: 3_aug1.jpg -> ASM=0.005037, Contrast=52.051954
处理完成: 3_aug2.jpg -> ASM=0.009751, Contrast=2.353877
处理完成: 4.jpg -> ASM=0.005582, Contrast=6.397769
处理完成: 4_aug1.jpg -> ASM=0.007666, Contrast=4.099849
处理完成: 4_aug2.jpg -> ASM=0.007695, Contrast=4.306212
处理完成: 5.jpg -> ASM=0.007896, Contrast=3.597125
处理完成: 5_aug1.jpg -> ASM=0.007709, Contrast=3.647732
处理完成: 5_aug2.jpg -> ASM=0.008385, Contrast=5.673508
处理完成: 6.jpg -> ASM=0.012103, Contrast=3.147447
处理完成: 6_aug1.jpg -> ASM=0.007728, Contrast=9.039287
处理完成: 6_aug2.jpg -> ASM=0.015446, Contrast=2.135580
处理完成: 7.jpg -> ASM=0.008334, Contrast=3.158769
处理完成: 7_aug1.jpg -> ASM=0.010161, Contrast=1.8

## 特征评估

In [5]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.metrics import roc_auc_score
from scipy.stats import mannwhitneyu, ks_2samp
from sklearn.preprocessing import MinMaxScaler

# ================================
# 1. 读取数据并统一ID格式
# ================================
# 读取特征表（假设第一列为 filename，后面为14个特征）
df_feat = pd.read_csv("glcm_features_full_processed.csv")

# 提取基础ID（去掉 .jpg 扩展名）
df_feat['image_id'] = df_feat.iloc[:, 0].str.replace('.jpg', '', regex=False)

# 读取标签表（假设第一列为 image_id，第二列为原始类别）
df_label = pd.read_csv("label.csv")
df_label.rename(columns={df_label.columns[0]: 'image_id', df_label.columns[1]: 'label'}, inplace=True)

# 合并（内连接，确保两个表都存在该图像）
df = pd.merge(df_feat, df_label, on='image_id', how='inner')
print(f"合并后样本数: {len(df)}")  # 应该为600

# 提取特征矩阵和标签
# 假设特征列是除了 filename 和 image_id 之外的所有列
feature_cols = [c for c in df_feat.columns if c not in ['filename', 'image_id']]  # 根据实际列名调整
X = df[feature_cols].values
y_raw = df['label'].values

# 将标签转换为二分类：vasc=1，其他=0
y = np.where(y_raw == 'vasc', 1, 0)
print(f"正类(vasc)样本数: {y.sum()}, 负类样本数: {len(y)-y.sum()}")

# ================================
# 2. 定义评估函数（含自定义FDR）
# ================================
def bh_fdr_correction(pvalues, alpha=0.05):
    """Benjamini-Hochberg FDR correction"""
    pvalues = np.asarray(pvalues)
    n = len(pvalues)
    order = np.argsort(pvalues)
    ranked_p = pvalues[order]
    adjusted = ranked_p * n / (np.arange(1, n + 1))
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0, 1)
    adjusted_pvalues = np.empty_like(adjusted)
    adjusted_pvalues[order] = adjusted
    reject = adjusted_pvalues <= alpha
    return adjusted_pvalues, reject

def compute_feature_metrics(X, y, feature_names):
    X_clean = np.nan_to_num(X)
    f_vals, f_pvals = f_classif(X_clean, y)
    mi_vals = mutual_info_classif(X_clean, y, random_state=42)

    rows = []
    vasc_idx = (y == 1)
    nonvasc_idx = (y == 0)

    for j, feat in enumerate(feature_names):
        v0 = X_clean[nonvasc_idx, j]
        v1 = X_clean[vasc_idx, j]

        # AUC
        try:
            auc = roc_auc_score(y, X_clean[:, j])
            auc = max(auc, 1 - auc)
        except:
            auc = 0.5

        # Cohen's d
        pooled_std = np.sqrt((v0.std()**2 + v1.std()**2) / 2)
        cohens_d = abs(v1.mean() - v0.mean()) / (pooled_std + 1e-9)

        # Mann-Whitney U p-value
        try:
            _, mw_p = mannwhitneyu(v0, v1, alternative='two-sided')
        except:
            mw_p = 1.0

        # KS 统计量
        try:
            ks_stat, _ = ks_2samp(v0, v1)
        except:
            ks_stat = 0.0

        rows.append({
            'feature': feat,
            'F_score': f_vals[j],
            'F_pvalue': f_pvals[j],
            'MI': mi_vals[j],
            'AUC': auc,
            'Cohens_d': cohens_d,
            'MW_pvalue': mw_p,
            'KS_stat': ks_stat,
        })

    df_scores = pd.DataFrame(rows)

    # 综合得分
    scaler = MinMaxScaler()
    df_scores['neg_log_MWp'] = -np.log10(df_scores['MW_pvalue'] + 1e-300)
    rank_cols = ['F_score', 'MI', 'AUC', 'Cohens_d', 'KS_stat', 'neg_log_MWp']
    normed = scaler.fit_transform(df_scores[rank_cols])
    df_scores['composite_score'] = normed.mean(axis=1)

    df_scores = df_scores.sort_values('composite_score', ascending=False).reset_index(drop=True)
    return df_scores

# ================================
# 3. 计算并输出结果
# ================================
df_scores = compute_feature_metrics(X, y, feature_cols)

# FDR 校正
pvals = df_scores['MW_pvalue'].values
p_adjusted, reject = bh_fdr_correction(pvals, alpha=0.05)
df_scores['MW_pvalue_FDR'] = p_adjusted
df_scores['FDR_significant'] = reject

# 强单变量特征
df_scores['is_strong_univariate'] = (
    (df_scores['AUC'] >= 0.70) &
    (df_scores['Cohens_d'] >= 0.80) &
    (df_scores['FDR_significant'])
)

# 显示结果
display_cols = ['feature', 'F_score', 'MI', 'AUC', 'Cohens_d', 'KS_stat',
                'MW_pvalue', 'MW_pvalue_FDR', 'FDR_significant', 'is_strong_univariate', 'composite_score']
print(df_scores[display_cols].to_string(index=False))

# 可选：保存结果
df_scores.to_csv("feature_ranking.csv", index=False)

合并后样本数: 600
正类(vasc)样本数: 90, 负类样本数: 510
           feature   F_score       MI      AUC  Cohens_d  KS_stat  MW_pvalue  MW_pvalue_FDR  FDR_significant  is_strong_univariate  composite_score
        SumEntropy  7.568157 0.086711 0.650654  0.353192 0.380392   0.000005       0.000071             True                 False         0.930822
          Variance 10.567077 0.057212 0.621394  0.404474 0.317647   0.000238       0.001111             True                 False         0.818557
           Entropy  7.270935 0.041999 0.640087  0.312811 0.312418   0.000022       0.000156             True                 False         0.750950
       SumVariance  7.147717 0.032736 0.591351  0.333338 0.266667   0.005689       0.015931             True                 False         0.577661
               ASM  1.063494 0.043493 0.608279  0.112800 0.277124   0.001047       0.003664             True                 False         0.458434
       Homogeneity  3.394005 0.000000 0.575338  0.209039 0.177124   0.02